# Pairs fOU 01 — Stage A: the thesis pipeline at 5 minutes on crypto perpetuals

Protocol: `discussions/Pairs_fOU/001_claude_stage_a_preregistration.md`. Source: García Petrov (2026), *Fractional Ornstein–Uhlenbeck Statistical Arbitrage with Synthetic Equity Options*.

Rolling 7-day formation (Engle–Granger 1%, $H<0.5$ by change-of-frequency, fOU moments, structural $T_{70}$), conditional first-passage $T_{70}\le 48$ bars from the last 60 inferred fGn innovations, $|Z|\ge1.5$, next-bar execution, exit at convergence or $T_{70}$. Perpetual legs, 12 bp round trip of gross notional. Benchmarks: OU ($H=0.5$) and a plain z-score rule.

In [1]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import fou_pairs as fp
importlib.reload(fp);assert fp.VERSION.startswith('PF01-0.1')
spec=fp.Spec()
SYNTHETIC=os.getenv('PF01_SYNTHETIC','0')=='1'
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='pf01-');OUT=Path(tmp.name)/'01';spec=fp.Spec(paths=1000,structural_paths=500,n_boot=500)
else:
    OUT=ROOT/'notebooks'/'Pairs_fOU'/'01'
OUT.mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_fou_pairs.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
print(chk.stdout[-1200:]);print(chk.stderr[-600:]);assert chk.returncode==0,'tests failed'
print(json.dumps(fp.registration(spec),indent=1))

.......                                                                  [100%]
7 passed in 31.51s


{
 "version": "PF01-0.1.0",
 "assets": [
  "BTC",
  "ETH",
  "SOL",
  "XRP"
 ],
 "formation_bars": 2016,
 "entry_z": 1.5,
 "target_p": 0.7,
 "max_horizon": 48,
 "structural_horizon": 252,
 "memory": 60,
 "paths": 5000,
 "structural_paths": 2000,
 "coint_p": 0.01,
 "adf_p": 0.05,
 "cost_bp_per_side_leg": 6.0,
 "test_start": "2025-10-01",
 "n_boot": 4000,
 "seed": 42,
 "status": "HISTORICAL_EXPLORATORY"
}


In [2]:
if SYNTHETIC:
    P=fp.synthetic_prices(n_days=14,seed=7,kappa=0.04,H=0.4)
    PAIRS=[('AAA','BBB')]
else:
    BIN=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent/'Binance'
    S={a:fp.log_close_5m(pd.read_parquet(BIN/f'{a}USDT_um_1m.parquet',columns=['open_time','close'])) for a in spec.assets}
    P=fp.panel(S);PAIRS=fp.pairs(spec.assets)
print('panel',P.shape,P.index.min(),'->',P.index.max(),'| pairs',PAIRS)
print('complete-bar share:',P.notna().mean().round(4).to_dict())

panel (213408, 4) 2024-10-01 00:05:00+00:00 -> 2026-10-12 00:00:00+00:00 | pairs [('BTC', 'ETH'), ('BTC', 'SOL'), ('BTC', 'XRP'), ('ETH', 'SOL'), ('ETH', 'XRP'), ('SOL', 'XRP')]
complete-bar share: {'BTC': 1.0, 'ETH': 1.0, 'SOL': 1.0, 'XRP': 1.0}


## Backtest (fOU, OU, z-score) — formation screening shared across modes

In [3]:
T0=time.time();cache={};TR=[];FC=[];FI=[]
for (y,x) in PAIRS:
    for mode in ('fou','ou','zscore'):
        tr,fc,fi=fp.run_pair(P,y,x,spec,mode=mode,screen_cache=cache)
        TR.append(tr);FC.append(fc);FI.append(fi)
        print(f'{y}-{x} {mode:6s} trades={len(tr):5d} forecasts={len(fc):6d} eligible days={int(fi.eligible.sum()) if len(fi) else 0:4d} | {time.time()-T0:7.0f}s',flush=True)
trades=pd.concat([t for t in TR if len(t)],ignore_index=True) if any(len(t) for t in TR) else pd.DataFrame()
fcs=pd.concat([f for f in FC if len(f)],ignore_index=True) if any(len(f) for f in FC) else pd.DataFrame()
fits=pd.concat(FI,ignore_index=True)

BTC-ETH fou    trades=    0 forecasts=   465 eligible days=   6 |     936s
BTC-ETH ou     trades=    0 forecasts=   553 eligible days=   8 |     939s
BTC-ETH zscore trades=   28 forecasts=    28 eligible days=   8 |     939s
BTC-SOL fou    trades=    0 forecasts=   976 eligible days=  10 |    1922s
BTC-SOL ou     trades=    9 forecasts=  1998 eligible days=  18 |    1937s
BTC-SOL zscore trades=   68 forecasts=    68 eligible days=  18 |    1938s
BTC-XRP fou    trades=   15 forecasts=   702 eligible days=  11 |    2943s
BTC-XRP ou     trades=   15 forecasts=   956 eligible days=  14 |    2951s
BTC-XRP zscore trades=   44 forecasts=    44 eligible days=  14 |    2951s
ETH-SOL fou    trades=    0 forecasts=  2201 eligible days=  17 |    4130s
ETH-SOL ou     trades=    0 forecasts=  2334 eligible days=  19 |    4146s
ETH-SOL zscore trades=   72 forecasts=    72 eligible days=  20 |    4146s
ETH-XRP fou    trades=   10 forecasts=   888 eligible days=  10 |    5624s
ETH-XRP ou     trades=   

In [4]:
print('Formation funnel (pair-days):')
fu=fits.groupby(['mode','reason']).size().unstack(fill_value=0);display(fu)
f1=fits[fits['mode']=='fou']
print('fOU eligible pair-days by pair:');display(f1.groupby('pair').agg(days=('day','size'),coint=('coint_ok','mean'),eligible=('eligible','mean'),
      H_median=('H','median'),half_life_bars=('half_life_bars','median')).round(3))

Formation funnel (pair-days):


reason,,H>=0.5,no_coint,structural
mode,,,,
fou,64,22,4314,4
ou,88,0,4314,2
zscore,90,0,4314,0


fOU eligible pair-days by pair:


,days,coint,eligible,H_median,half_life_bars
pair,,,,,
BTC-ETH,734,0.011,0.008,0.439,76.362
BTC-SOL,734,0.025,0.014,0.485,48.306
BTC-XRP,734,0.019,0.015,0.481,50.175
ETH-SOL,734,0.027,0.023,0.475,52.561
ETH-XRP,734,0.020,0.014,0.485,50.853
SOL-XRP,734,0.020,0.014,0.493,45.470


In [5]:
TS=pd.Timestamp(spec.test_start,tz='UTC')
if len(trades):
    trades['period']=np.where(trades.entry_bar>=TS,'test','dev')
rows=[]
for mode in ('fou','ou','zscore'):
    for per in ('dev','test'):
        t=trades[(trades['mode']==mode)&(trades.period==per)] if len(trades) else trades
        rows.append(fp.trade_summary(t,spec,f'{mode}|{per}|all'))
        for pr,g in (t.groupby('pair') if len(t) else []):
            rows.append(fp.trade_summary(g,spec,f'{mode}|{per}|{pr}'))
summ=pd.DataFrame(rows);display(summ.round(3))
cal=[]
for mode in ('fou','ou'):
    for per in ('dev','test'):
        f=fcs[(fcs['mode']==mode)] if len(fcs) else fcs
        if len(f):f=f[(f.signal_bar>=TS)==(per=='test')]
        cal.append({'mode':mode,'period':per,**fp.calibration(f)})
cal=pd.DataFrame(cal);print('Convergence calibration (thesis: 70.3% predicted vs 33.1% realised):');display(cal.round(4))

,label,trades,days_with_trades,gross_bp,gross_lo,gross_hi,net_bp,net_lo,net_hi,win_rate_net,converge_share,mean_bars_held
0,fou|dev|all,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,fou|test|all,25,4.0,11.833,-1.676,16.360,-0.167,-13.676,4.360,0.520,0.000,24.400
2,fou|test|BTC-XRP,15,3.0,11.375,6.646,16.692,-0.625,-5.354,4.692,0.533,0.000,23.933
3,fou|test|ETH-XRP,10,2.0,12.520,-10.073,27.582,0.520,-22.073,15.582,0.500,0.000,25.100
4,ou|dev|all,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ou|test|all,35,5.0,6.363,3.420,11.329,-5.637,-8.580,-0.671,0.429,0.029,24.057
6,ou|test|BTC-SOL,9,2.0,-0.299,-35.440,27.814,-12.299,-47.440,15.814,0.333,0.111,22.333
7,ou|test|BTC-XRP,15,3.0,10.188,7.266,13.966,-1.812,-4.734,1.966,0.533,0.000,23.600
8,ou|test|ETH-XRP,11,3.0,6.596,-14.385,31.164,-5.404,-26.385,19.164,0.364,0.000,26.091
9,zscore|dev|all,79,20.0,-0.451,-5.473,3.946,-12.451,-17.473,-8.054,0.304,0.076,45.215


Convergence calibration (thesis: 70.3% predicted vs 33.1% realised):


,mode,period,forecasts,complete,mean_predicted,observed,brier
0,fou,dev,0,NaN,NaN,NaN,NaN
1,fou,test,25,25.0,0.7093,0.0000,0.5031
2,ou,dev,0,NaN,NaN,NaN,NaN
3,ou,test,35,35.0,0.7097,0.0286,0.4917


In [6]:
dec=fp.decision(summ,synthetic=SYNTHETIC);print(json.dumps(dec,indent=2,default=float))
if len(trades):trades.to_csv(OUT/'01_trades.csv',index=False,float_format='%.6g')
fits.drop(columns=[c for c in ('resid',) if c in fits]).to_csv(OUT/'01_formation.csv',index=False,float_format='%.6g')
summ.to_csv(OUT/'01_summary.csv',index=False);cal.to_csv(OUT/'01_calibration.csv',index=False)
(OUT/'01_decision.json').write_text(json.dumps(dec,indent=2,default=float),encoding='utf-8')
(OUT/'01_manifest.json').write_text(json.dumps({'registration':fp.registration(spec),'synthetic':SYNTHETIC,
   'panel':[str(P.index.min()),str(P.index.max())],'runtime_s':time.time()-T0},indent=2,default=str),encoding='utf-8')
print('saved',sorted(p.name for p in OUT.iterdir()))

{
  "verdict": "INCONCLUSIVE",
  "stage_b_unlocked": false,
  "test_trades": 25,
  "gross_bp": [
    11.832744604763935,
    -1.6760680067537932,
    16.359992891313265
  ],
  "net_bp": [
    -0.16725539523606586,
    -13.676068006753793,
    4.359992891313264
  ],
  "status": "HISTORICAL_EXPLORATORY"
}
saved ['01_calibration.csv', '01_decision.json', '01_formation.csv', '01_manifest.json', '01_summary.csv', '01_trades.csv']
